In [2]:
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications.resnet50 import preprocess_input
from ultralytics import YOLO
from boxmot import BotSort
from pathlib import Path
import warnings

In [3]:
SEGMENTATION_MODEL_PATH = './bestmodel.keras'
REID_WEIGHTS_PATH = Path('osnet_x1_0_msmt17.pt')

YOLO_MODEL_PATH = "./best.pt"

In [4]:
IMG_SIZE = (256, 256)

In [5]:
def preprocess_frame_for_segmentation(frame, img_size):
    img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    img_tensor = tf.convert_to_tensor(img_rgb, dtype=tf.float32)
    img_resized = tf.image.resize(img_tensor, img_size)
    img_preprocessed = preprocess_input(img_resized)
    img_for_prediction = tf.expand_dims(img_preprocessed, axis=0)
    return img_for_prediction

In [7]:
segmentation_model = tf.keras.models.load_model(SEGMENTATION_MODEL_PATH, compile=False)

In [8]:
yolo_model = YOLO(YOLO_MODEL_PATH)

In [9]:
vehicle_tracker = BotSort(
    reid_weights=REID_WEIGHTS_PATH,
    device='0',
    half=False,
    with_reid=True
)

2025-09-24 08:45:40.339 | INFO     | boxmot.utils.torch_utils:select_device:78 - Yolo Tracking v15.0.1 🚀 Python-3.10.12 torch-2.7.1+cu126
CUDA:0 (NVIDIA GeForce RTX 3070 Laptop GPU, 8192MiB)
2025-09-24 08:45:40.341 | ERROR    | boxmot.appearance.backends.base_backend:download_model:152 - Found existing ReID weights at osnet_x1_0_msmt17.pt; skipping download.
2025-09-24 08:45:40.572 | SUCCESS  | boxmot.appearance.reid.registry:load_pretrained_weights:64 - Loaded pretrained weights from osnet_x1_0_msmt17.pt


In [11]:
VIDEO_PATH = "video.mp4"

cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    print(f"Error: Could not open video file {VIDEO_PATH}")
    exit()

frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # --- مرحله ۱: سگمنت‌بندی خطوط ---
    input_tensor = preprocess_frame_for_segmentation(frame, IMG_SIZE)
    prediction_logits = segmentation_model.predict(input_tensor, verbose=0)
    output_mask_tensor = tf.argmax(prediction_logits, axis=-1)
    output_mask = tf.squeeze(output_mask_tensor, axis=0).numpy().astype(np.uint8)
    output_mask_resized = cv2.resize(output_mask, (frame_width, frame_height), interpolation=cv2.INTER_NEAREST)

    # --- مرحله ۲: تشخیص و ردیابی خودروها ---
    results = yolo_model(frame, stream=True, verbose=False)
    
    detections_for_tracker = []
    for result in results:
        boxes = result.boxes.xyxy.cpu().numpy()
        scores = result.boxes.conf.cpu().numpy()
        labels = result.boxes.cls.cpu().numpy()
        
        if boxes.size > 0:
            detections = np.hstack((boxes, scores[:, np.newaxis], labels[:, np.newaxis]))
            detections_for_tracker.extend(detections)

    tracks = []
    if detections_for_tracker:
        tracks = vehicle_tracker.update(np.array(detections_for_tracker), frame)

    # --- مرحله ۳: بصری‌سازی نتایج ---
    color_mask = np.zeros_like(frame)
    color_mask[output_mask_resized == 1] = [0, 0, 255] 
    combined_frame = cv2.addWeighted(frame, 0.7, color_mask, 0.3, 0)
    
    if len(tracks) > 0:
        for track in tracks:
            x1, y1, x2, y2, track_id, _, _, _ = track
            x1, y1, x2, y2 = map(int, [x1, y1, x2, y2])
            track_id = int(track_id)
            
            cv2.rectangle(combined_frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(combined_frame, f"ID: {track_id}", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

    cv2.imshow("Lane Segmentation and Vehicle Tracking", combined_frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()